# Introduction

https://www.kaggle.com/snap/amazon-fine-food-reviews
The dataset consists of 13 year’s data which consists of 10 attributes for 568000 reviews. Due to the computational complexity, we will use a random sample of 10,000 reviews for our analysis.

Lets understand on how and what kind of  clusters are formed based on text reviews. Please note we are not performing DENSITY BASED for bag of words and TF-IDF as it is not feasible with lot of features or columns.However, with Avg word to vectors, the DBSCAN is shown in the notebook,

The dataset contains the following columns :

1.Id->Review for each ID

2.Product Id->Unique identifier for the product

3.User Id->Unique identifier for the user

4.Profile Name->A user who has given the review

5.Helpful Numerator->No. of users who found the review helpful

6.Helpful Denominator->No. of users who found the review helpful or not

7.Score->Five being is the highest rating and 1 being the lowest rating

8.Time->Date and time  when the review was given

9.Summary->Summary of the review

10.Text->Review text


In [ ]:
import pandas as pd
#load the data
sample=pd.read_csv("../input/samplereviews.csv")

In [ ]:
#check the loaded data
print(sample.shape)

In [ ]:
#look of the dataset
sample.head()

In [ ]:
# Understand how customer ratings are distributed
import seaborn as sns
sns.countplot(sample.Score)

 # Data Cleaning

In [ ]:
#converting the Numerical reviws to categorical reviews on codition above 3 are
#positive and below 3 are negative as reviews rating with 3 are not much useful
#for analysis

#function
def partition(x):
    if x < 3:
        return 'negative'
    return 'positive'

#changing reviews with score less than 3 to be positive
actualScore = sample['Score']
positiveNegative = actualScore.map(partition) 
sample['Score'] = positiveNegative

In [ ]:
sample.head()

In [ ]:
# no of positive and negative reviews
sample["Score"].value_counts()
#here we can say it is a unbalanced data set

In [ ]:
#dropping  the duplicates column if any using drop duplicates from pandas
sorted_data=sample.sort_values('ProductId', axis=0, ascending=True, inplace=False, kind='quicksort', na_position='last')
final=sorted_data.drop_duplicates(subset={"UserId","ProfileName","Time","Text"}, keep='first', inplace=False)
final.shape

In [ ]:
# no duplicate columns found
(final['Id'].size*1.0)/(sample['Id'].size*1.0)*100

In [ ]:
final=final[final.HelpfulnessNumerator<=final.HelpfulnessDenominator]
# Help..Num is always less than Denom.. as Denom is people who upvote and donwvote
#Before understanding text preprocessing lets see the number of entries left
print(final.shape)

#How many positive and negative reviews are present in our dataset?
final['Score'].value_counts()

# after removing duplicate rows we found, 8346 positive and 1457 negative

In [ ]:
# After Removing Duplicate rows
import seaborn as sns
sns.countplot(final.Score)

# Text Processing

To make the text clean by removing HTML tag reviews, stopwords to segregate and adding timestamp

In [ ]:
# find sentences containing HTML tags
import re
i=0;
for sent in final['Text'].values:
    if (len(re.findall('<.*?>', sent))):
        print(i)
        print(sent)
        break;
    i += 1;

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
sno = nltk.stem.SnowballStemmer('english') #initialising the snowball stemmer which is developed in recent years
stop=set(stopwords.words('english'))



def cleanhtml(sentence): #function to clean the word of any html-tags
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, ' ', sentence)
    return cleantext
def cleanpunc(sentence): #function to clean the word of any punctuation or special characters
    cleaned = re.sub(r'[?|!|\'|"|#]',r'',sentence)
    cleaned = re.sub(r'[.|,|)|(|\|/]',r' ',cleaned)
    return  cleaned
print(stop)
print('************************************')
print(sno.stem('tasty'))

In [ ]:
i=0
str1=' '
final_string=[]
all_positive_words=[] # store words from +ve reviews here
all_negative_words=[] # store words from -ve reviews here.
s=''
for sent in final['Text'].values:
    filtered_sentence=[]
    #print(sent);
    sent=cleanhtml(sent) # remove HTMl tags
    for w in sent.split():
        for cleaned_words in cleanpunc(w).split():
            if((cleaned_words.isalpha()) & (len(cleaned_words)>2)):    
                if(cleaned_words.lower() not in stop):
                    s=(sno.stem(cleaned_words.lower())).encode('utf8')
                    filtered_sentence.append(s)
                    if (final['Score'].values)[i] == 'positive': 
                        all_positive_words.append(s) #list of all words used to describe positive reviews
                    if(final['Score'].values)[i] == 'negative':
                        all_negative_words.append(s) #list of all words used to describe negative reviews reviews
                else:
                    continue
            else:
                continue 
    #print(filtered_sentence)
    str1 = b" ".join(filtered_sentence) #final string of cleaned words
    #print("***********************************************************************")
    
    final_string.append(str1)
    i+=1

In [ ]:
final['CleanedText']=final_string #adding a column of CleanedText which displays the data after pre-processing of the review 
final['CleanedText']=final['CleanedText'].str.decode("utf-8")

In [ ]:
final.shape # cleaned text column added

In [ ]:
final.head(3) #below the processed review can be seen in the CleanedText Column 

In [ ]:
# however, this is not required for clustering, just segregaring positive,negative and storing seperetly
data_pos = final[final["Score"] == "positive"]
data_neg = final[final["Score"] == "negative"]
final = pd.concat([data_pos, data_neg])
score =final["Score"]
final.head()


In [ ]:
#Converting the time frame and sorting in increasing order for easyness
final["Time"] = pd.to_datetime(final["Time"], unit = "s")
final= final.sort_values(by = "Time")
final.head()

#  Clustering

Find Clustering models for both Bag of words, term frequcny/ inverse document frequcny and avg word to vector

### K means using bag of words

In [ ]:
# Generating bag of words features.
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
bow = count_vect.fit_transform(final['CleanedText'].values)
bow.shape

In [ ]:
bow

In [ ]:
# to understand what kind of words generated as columns by BOW
terms = count_vect.get_feature_names()

In [ ]:
#first 10 columns generated by BOW
terms[1:10]

In [ ]:
#using all processes jobs=-1 and k means++ for starting initilization advantage
from sklearn.cluster import KMeans
model = KMeans(n_clusters = 10,init='k-means++', n_jobs = -1,random_state=99)
model.fit(bow)

In [ ]:
labels = model.labels_
cluster_center=model.cluster_centers_

In [ ]:
cluster_center

In [ ]:
from sklearn import metrics
silhouette_score = metrics.silhouette_score(bow, labels, metric='euclidean')

In [ ]:
# which tells us that clusters are far away from each other 
silhouette_score

In [ ]:
# Giving Labels/assigning a cluster to each point/text 
df = final
df['Bow Clus Label'] = model.labels_ # the last column you can see the label numebers
df.head(2)

In [ ]:
# How many points belong to each cluster -> using group by in pandas
df.groupby(['Bow Clus Label'])['Text'].count()

In [ ]:
#Refrence credit - to find the top 10 features of cluster centriod
#https://stackoverflow.com/questions/47452119/kmean-clustering-top-terms-in-cluster
print("Top terms per cluster:")
order_centroids = model.cluster_centers_.argsort()[:, ::-1]
terms = count_vect.get_feature_names()
for i in range(10):
    print("Cluster %d:" % i, end='')
    for ind in order_centroids[i, :10]:
        print(' %s' % terms[ind], end='')
        print()

In [ ]:
# visually how points or reviews are distributed across 10 clusters 
import matplotlib.pyplot as plt
plt.bar([x for x in range(10)], df.groupby(['Bow Clus Label'])['Text'].count(), alpha = 0.4)
plt.title('KMeans cluster points')
plt.xlabel("Cluster number")
plt.ylabel("Number of points")
plt.show()

In [ ]:
# Reading a review which belong to each group.
for i in range(10):
    print("A review of assigned to cluster ", i)
    print("-" * 70)
    print(df.iloc[df.groupby(['Bow Clus Label']).groups[i][0]]['Text'])
    print('\n')
    print("_" * 70)

In [ ]:
#considers sample of 3 random reviews for cluster 0

print(df.iloc[df.groupby(['Bow Clus Label']).groups[0][3]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[0][15]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[0][25]]['Text'])

In [ ]:
#consider sample of 3 random reviews for cluster 4

print(df.iloc[df.groupby(['Bow Clus Label']).groups[3][3]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[3][15]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[3][25]]['Text'])

In [ ]:
#consider sample of 3 random reviews for cluster 4

print(df.iloc[df.groupby(['Bow Clus Label']).groups[5][3]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[5][15]]['Text'])
print("_" * 70)
print(df.iloc[df.groupby(['Bow Clus Label']).groups[5][25]]['Text'])

__Analysis of K means for BOW:__

__Of all the clusters, 0, 4 and 6 accounts to more % of reviews, undertsanding differences between these 3 clusters is key.
Also, the clusters 2 and 9 have only 1 review__ 

If we observe the top terms per cluster, The cluster 4 which consists of LIKE AND LOVE, which are top centroid features and can say this cluster consists of all positive reviews, let us obersve few reviews of each cluster and try to understand the differences

By reading the cluster 2 and 9 which contains only one review, which is clearlt negative reviews and we can conluded customers
didnt liked the product at all and not word is used extensively

By reading random reviews of cluster 0, we can easily say that these reviews are extremly positive of the product usage and customers are very happy with the product

By reading random reviews of cluster 4, we can  say that the key word __BUT__ is repeating acorss the review which indicates some kind of peopel agree with most of the things related to the products but their is something which is slightyly disagree with product quality or delivery or  some thing less than their expectation



##  K means using TFIDF

In [ ]:
#tfidf vector initililization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer()
tfidf = tfidf_vect.fit_transform(final['CleanedText'].values)
tfidf.shape

In [ ]:
from sklearn.cluster import KMeans
model_tf = KMeans(n_clusters = 10, n_jobs = -1,random_state=99)
model_tf.fit(tfidf)

In [ ]:
labels_tf = model_tf.labels_
cluster_center_tf=model_tf.cluster_centers_

In [ ]:
cluster_center_tf

In [ ]:
# to understand what kind of words generated as columns by BOW
terms1 = tfidf_vect.get_feature_names()

In [ ]:
terms1[1:10]

In [ ]:
from sklearn import metrics
silhouette_score_tf = metrics.silhouette_score(tfidf, labels_tf, metric='euclidean')

In [ ]:
silhouette_score_tf

In [ ]:
# Giving Labels/assigning a cluster to each point/text 
df1 = df
df1['Tfidf Clus Label'] = model_tf.labels_
df1.head(5)

In [ ]:
# How many points belong to each cluster ->

df1.groupby(['Tfidf Clus Label'])['Text'].count()

In [ ]:
#Refrence credit - to find the top 10 features of cluster centriod
#https://stackoverflow.com/questions/47452119/kmean-clustering-top-terms-in-cluster
print("Top terms per cluster:")
order_centroids = model_tf.cluster_centers_.argsort()[:, ::-1]
for i in range(10):
    print("Cluster %d:" % i, end='')
    for ind in order_centroids[i, :10]:
        print(' %s' % terms1[ind], end='')
        print()

In [ ]:
# visually how points or reviews are distributed across 10 clusters 

plt.bar([x for x in range(10)], df1.groupby(['Tfidf Clus Label'])['Text'].count(), alpha = 0.4)
plt.title('KMeans cluster points')
plt.xlabel("Cluster number")
plt.ylabel("Number of points")
plt.show()

In [ ]:
# Reading a review which belong to each group.
for i in range(10):
    print("4 review of assigned to cluster ", i)
    print("-" * 70)
    print(df1.iloc[df1.groupby(['Tfidf Clus Label']).groups[i][5]]['Text'])
    print('\n')
    print(df1.iloc[df1.groupby(['Tfidf Clus Label']).groups[i][10]]['Text'])
    print('\n')
    print(df1.iloc[df1.groupby(['Tfidf Clus Label']).groups[i][20]]['Text'])
    print('\n')
    print("_" * 70)

__Analysis of K means for TF_IDF:__

__Of all the cluster 4  accounts to more % of reviews i,e above 4000.

If we observe the top terms per cluster, The clusters based on the products and product wise like and dislikes.
for example, if we oberve cluster 8, the reviews talk more about chips, potatos, and other products which are like snacks

In these, its better to understand the cluster center top features rather than individual reviews.

# Average Word to Vector

In [ ]:
# Train your own Word2Vec model using your own text corpus
i=0
list_of_sent=[]
for sent in final['CleanedText'].values:
    list_of_sent.append(sent.split())

In [ ]:
print(final['CleanedText'].values[0])
print("*****************************************************************")
print(list_of_sent[0])

In [ ]:

# removing html tags and apostrophes if present.
import re
def cleanhtml(sentence): #function to clean the word of any html-tags
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, ' ', sentence)
    return cleantext
def cleanpunc(sentence): #function to clean the word of any punctuation or special characters
    cleaned = re.sub(r'[?|!|\'|"|#]',r'',sentence)
    cleaned = re.sub(r'[.|,|)|(|\|/]',r' ',cleaned)
    return  cleaned

In [ ]:
i=0
list_of_sent_train=[]
for sent in final['CleanedText'].values:
    filtered_sentence=[]
    sent=cleanhtml(sent)
    for w in sent.split():
        for cleaned_words in cleanpunc(w).split():
            if(cleaned_words.isalpha()):    
                filtered_sentence.append(cleaned_words.lower())
            else:
                continue 
    list_of_sent_train.append(filtered_sentence)

In [ ]:
import gensim
# Training the wor2vec model using train dataset
w2v_model=gensim.models.Word2Vec(list_of_sent_train,size=100, workers=4)

In [ ]:
import numpy as np
sent_vectors = []; # the avg-w2v for each sentence/review is stored in this train
for sent in list_of_sent_train: # for each review/sentence
    sent_vec = np.zeros(100) # as word vectors are of zero length
    cnt_words =0; # num of words with a valid vector in the sentence/review
    for word in sent: # for each word in a review/sentence
        try:
            vec = w2v_model.wv[word]
            sent_vec += vec
            cnt_words += 1
        except:
            pass
    sent_vec /= cnt_words
    sent_vectors.append(sent_vec)
sent_vectors = np.array(sent_vectors)
sent_vectors = np.nan_to_num(sent_vectors)
sent_vectors.shape


## K Means CLustering for Avg word to vectors

In [ ]:
# Number of clusters to check.
num_clus = [x for x in range(3,11)]
num_clus

In [ ]:
# Choosing the best cluster using Elbow Method.
# source credit,few parts of min squred loss info is taken from different parts of the stakoverflow answers.
# this is used to understand to find the optimal clusters in differen way rather than used in BOW, TFIDF
squared_errors = []
for cluster in num_clus:
    kmeans = KMeans(n_clusters = cluster).fit(sent_vectors) # Train Cluster
    squared_errors.append(kmeans.inertia_) # Appending the squared loss obtained in the list
    
optimal_clusters = np.argmin(squared_errors) + 2 # As argmin return the index of minimum loss. 
plt.plot(num_clus, squared_errors)
plt.title("Elbow Curve to find the no. of clusters.")
plt.xlabel("Number of clusters.")
plt.ylabel("Squared Loss.")
xy = (optimal_clusters, min(squared_errors))
plt.annotate('(%s, %s)' % xy, xy = xy, textcoords='data')
plt.show()

print ("The optimal number of clusters obtained is - ", optimal_clusters)
print ("The loss for optimal cluster is - ", min(squared_errors))

In [ ]:
# Training the best model --
from sklearn.cluster import KMeans
model2 = KMeans(n_clusters = optimal_clusters)
model2.fit(sent_vectors)

In [ ]:
word_cluster_pred=model2.predict(sent_vectors)
word_cluster_pred_2=model2.labels_
word_cluster_center=model2.cluster_centers_

In [ ]:
word_cluster_center[1:2]

In [ ]:
# Giving Labels/assigning a cluster to each point/text 
dfa = df1
dfa['AVG-W2V Clus Label'] = model2.labels_
dfa.head(2)

In [ ]:
# How many points belong to each cluster ->
dfa.groupby(['AVG-W2V Clus Label'])['Text'].count()

In [ ]:
# Reading a review which belong to each group.
for i in range(optimal_clusters):
    print("A review of assigned to cluster ", i)
    print("-" * 70)
    print(dfa.iloc[dfa.groupby(['AVG-W2V Clus Label']).groups[i][0]]['Text'])
    print('\n')
    print(dfa.iloc[dfa.groupby(['AVG-W2V Clus Label']).groups[i][1]]['Text'])
    print('\n')
    print("_" * 70)

## Clustering DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
# Computing 200th Nearest neighbour distance
minPts = 2 * 100
# Lower bound function copied from -> https://gist.github.com/m00nlight/0f9306b4d4e61ba0195f
def lower_bound(nums, target): # This function return the number in the array just greater than or equal to itself.
    l, r = 0, len(nums) - 1
    while l <= r: # Binary searching.
        mid = int(l + (r - l) / 2)
        if nums[mid] >= target:
            r = mid - 1
        else:
            l = mid + 1
    return l

def compute200thnearestneighbour(x, data): # Returns the distance of 200th nearest neighbour.
    dists = []
    for val in data:
        dist = np.sum((x - val) **2 ) # computing distances.
        if(len(dists) == 200 and dists[199] > dist): # If distance is larger than current largest distance found.
            l = int(lower_bound(dists, dist)) # Using the lower bound function to get the right position.
            if l < 200 and l >= 0 and dists[l] > dist:
                dists[l] = dist
        else:
            dists.append(dist)
            dists.sort()
    
    return dists[199] # Dist 199 contains the distance of 200th nearest neighbour.

In [ ]:
# Computing the 200th nearest neighbour distance of some point the dataset:
twohundrethneigh = []
for val in sent_vectors[:1500]:
    twohundrethneigh.append( compute200thnearestneighbour(val, sent_vectors[:1500]) )
twohundrethneigh.sort()

In [ ]:

# Plotting for the Elbow Method :
plt.figure(figsize=(14,4))
plt.title("Elbow Method for Finding the right Eps hyperparameter")
plt.plot([x for x in range(len(twohundrethneigh))], twohundrethneigh)
plt.xlabel("Number of points")
plt.ylabel("Distance of 200th Nearest Neighbour")
plt.show()


Conclusions for Elbow Method

The Knee point seems to be 5. So Eps = 5

In [ ]:
# Training DBSCAN :
model = DBSCAN(eps = 5, min_samples = minPts, n_jobs=-1)
model.fit(sent_vectors)

In [ ]:

dfdb = dfa
dfdb['AVG-W2V Clus Label'] = model.labels_
dfdb.head(2)

In [ ]:
dfdb.groupby(['AVG-W2V Clus Label'])['Id'].count()

# Clustering Hierarchical

In [ ]:
import scipy
from scipy.cluster import hierarchy
dendro=hierarchy.dendrogram(hierarchy.linkage(sent_vectors,method='ward'))
plt.axhline(y=35)# cut at 30 to get 5 clusters

In [ ]:
from sklearn.cluster import AgglomerativeClustering

cluster = AgglomerativeClustering(n_clusters=5, affinity='euclidean', linkage='ward')  #took n=5 from dendrogram curve 
Agg=cluster.fit_predict(sent_vectors)

In [ ]:
# Giving Labels/assigning a cluster to each point/text 
aggdfa = dfdb
aggdfa['AVG-W2V Clus Label'] = cluster.labels_
aggdfa.head(2)

In [ ]:
# How many points belong to each cluster ->
aggdfa.groupby(['AVG-W2V Clus Label'])['Text'].count()

In [ ]:
# Reading a review which belong to each group.
for i in range(5):
    print("2 reviews of assigned to cluster ", i)
    print("-" * 70)
    print(aggdfa.iloc[aggdfa.groupby(['AVG-W2V Clus Label']).groups[i][0]]['Text'])
    print('\n')
    print(aggdfa.iloc[aggdfa.groupby(['AVG-W2V Clus Label']).groups[i][1]]['Text'])
    print('\n')
    print("_" * 70)

Conclusion:
Kmeans for bag of words and TFIDF
1. By using Elbow method, we generated optimal 10 clusters for both the bag of words and tfidf techniques
2. In both the cases, one cluster accounts around 6000 reviews which is large chunk from 10k reviews and rest are distributed unevenly
3. we can ignore 2 clusters or keep 2 clusters depending upon the business goal for bag of words generation as both contain only 1 review

Final Observations:

FOR TFIDF K means is best for identification than K MEANS for BOW, all the clusters are clearly refelcting they were grouped based on the categories/products. However, K means did best on the cluster centers top terms but however when we caopare reviews , few places it is not correalting.

DBSCAN is very poorly performining on the 10k columns as it is grouping all reviews in one cluster

Hierarchical, for BOW and TFIDF, we cannot identify the clusters and not divded unevenly, but for avg word to vectors all are grouped and divided evenly. It is very difficult to identify the type of reveiws based on Hirarchial formation. (PLEASE NOTE  HIERARCHICAL  IS TAKING SO MUCH TIME TO PROCESS, Please if any one would like to see the hierarchical , please upvote and i will send the jupyter notebook) Thank you





